# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 03: Headers, Content-Type y Negociación de Contenido
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Lectura y emisión manual de encabezados HTTP*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Invariantes de negociación (Content-Type vs Accept)* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas con tipos de contenido no soportados (415)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion03.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [5]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
✅ Entorno Web listo:
Python 3.13.16
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Lectura y emisión manual de encabezados HTTP

In [10]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status, Request, Response
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest, request: Request, response: Response):
    # 1. LECTURA MANUAL DEL ENCABEZADO (Content-Type)
    content_type = request.headers.get("Content-Type", "")

    # Si el cliente envía algo que no es JSON, lo rechazamos
    if "application/json" not in content_type:
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail="El formato debe ser application/json"
        )

    # 2. Validación semántica de los datos
    if not payload.dato.strip():
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio"
        )

    # 3. EMISIÓN MANUAL DEL ENCABEZADO EN LA RESPUESTA
    # Nos aseguramos de decirle al cliente que le devolvemos un JSON
    response.headers["X-Auditoria"] = "Artesano-Sesion03"

    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [2]:
%%writefile app_ia.py
from fastapi import FastAPI, Request, HTTPException, status
from pydantic import BaseModel, ValidationError

app = FastAPI(title="API Científico - Sesión 03")

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
async def procesar_recurso(request: Request):
    # 1. Validación MANUAL del Content-Type (Antes de Pydantic)
    content_type = request.headers.get("Content-Type", "")

    if "application/json" not in content_type:
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail="El Content-Type debe ser application/json"
        )

    # 2. Leemos el cuerpo manualmente y lo validamos con Pydantic
    try:
        body = await request.json()
        payload = PayloadRequest(**body)
    except Exception:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="El cuerpo no es un JSON válido o no cumple con el esquema"
        )

    # 3. Validación semántica del dato
    if not payload.dato.strip():
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacío"
        )

    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# Pega aquí el código generado por la IA para auditarlo


Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas con tipos de contenido no soportados (415)

In [4]:
import threading
import time
import uvicorn
import httpx
from app_ia import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

# Levantar servidor
threading.Thread(target=run_server, daemon=True).start()
time.sleep(1)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== TEST HARNESS: SESIÓN 03 ===\n")

# Payload válido
r1 = client.post("/api/recurso", json={"dato": "Test Harness"})
print(f"Prueba 1 (JSON): {r1.status_code}")
print(f"Respuesta: {r1.json()}\n")

# Forzando Content-Type incorrecto para validar el 415
headers = {"Content-Type": "text/plain"}
r2 = client.post("/api/recurso", json={"dato": "Test Harness"}, headers=headers)
print(f"Prueba 2 (Header incorrecto): {r2.status_code}")
print(f"Respuesta: {r2.json()}\n")

if r1.status_code == 200 and r2.status_code == 415:
    print("Auditoría completada correctamente.")
else:
    print("Error en las pruebas.")

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


=== TEST HARNESS: SESIÓN 03 ===

Prueba 1 (JSON): 200
Respuesta: {'status': 'success', 'mensaje': 'Procesado: Test Harness'}

Prueba 2 (Header incorrecto): 415
Respuesta: {'detail': 'El Content-Type debe ser application/json'}

Auditoría completada correctamente.
